In [609]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing  import OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA

In [610]:
df = pd.read_csv('heart.csv')
df.head(6)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
5,39,M,NAP,120,339,0,Normal,170,N,0.0,Up,0


In [611]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [612]:
def outliers_remove_using_z_score(dataframe, col):
  z_score = (dataframe[col] - dataframe[col].mean()) / dataframe[col].std()
  return dataframe[(z_score > -3) & (z_score < 3)]

In [613]:
df1 = outliers_remove_using_z_score(df, 'Cholesterol')
df2 = outliers_remove_using_z_score(df1, 'RestingBP')
df3 = outliers_remove_using_z_score(df2, 'FastingBS')
df4 = outliers_remove_using_z_score(df3, 'MaxHR')
df5 = outliers_remove_using_z_score(df4, 'Oldpeak')

In [614]:
df5.sample(6)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
661,49,M,NAP,118,149,0,LVH,126,N,0.8,Up,1
405,35,M,ASY,120,0,1,Normal,130,Y,1.2,Flat,1
387,53,M,ASY,130,0,0,LVH,135,Y,1.0,Flat,1
115,33,F,ASY,100,246,0,Normal,150,Y,1.0,Flat,1
185,58,M,NAP,160,211,1,ST,92,N,0.0,Flat,1
664,65,F,ASY,150,225,0,LVH,114,N,1.0,Flat,1


In [615]:
df5.isna().sum()

,0
Age,0
Sex,0
ChestPainType,0
RestingBP,0
Cholesterol,0
FastingBS,0
RestingECG,0
MaxHR,0
ExerciseAngina,0
Oldpeak,0


In [616]:
df5.info()

<class 'pandas.core.frame.DataFrame'>
Index: 899 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             899 non-null    int64  
 1   Sex             899 non-null    object 
 2   ChestPainType   899 non-null    object 
 3   RestingBP       899 non-null    int64  
 4   Cholesterol     899 non-null    int64  
 5   FastingBS       899 non-null    int64  
 6   RestingECG      899 non-null    object 
 7   MaxHR           899 non-null    int64  
 8   ExerciseAngina  899 non-null    object 
 9   Oldpeak         899 non-null    float64
 10  ST_Slope        899 non-null    object 
 11  HeartDisease    899 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 91.3+ KB


In [617]:
df5.select_dtypes(include='object').columns

Index(['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope'], dtype='object')

In [618]:
df5.Sex.unique()

array(['M', 'F'], dtype=object)

In [619]:
df5.ExerciseAngina.unique()

array(['N', 'Y'], dtype=object)

In [620]:
df5.ChestPainType.unique()

array(['ATA', 'NAP', 'ASY', 'TA'], dtype=object)

In [621]:
df5.RestingECG.unique()

array(['Normal', 'ST', 'LVH'], dtype=object)

In [622]:
df5.ST_Slope.unique()

array(['Up', 'Flat', 'Down'], dtype=object)

In [623]:
df6 = df5.replace({
    'Sex': {'M': 0, 'F': 1},
    'ExerciseAngina': {'N': 0, 'Y': 1}
})
df6.sample(5)

/tmp/ipython-input-2520430501.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df6 = df5.replace({


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
480,58,0,ATA,126,0,1,Normal,110,1,2.0,Flat,1
728,54,1,NAP,135,304,1,Normal,170,0,0.0,Up,0
797,41,0,ASY,110,172,0,LVH,158,0,0.0,Up,1
187,41,0,ASY,120,237,1,Normal,138,1,1.0,Flat,1
254,55,0,ASY,145,248,0,Normal,96,1,2.0,Flat,1


In [624]:
df6.select_dtypes(include='object').columns

Index(['ChestPainType', 'RestingECG', 'ST_Slope'], dtype='object')

In [625]:
df6 = pd.get_dummies(df6, columns=['RestingECG', 'ST_Slope'], drop_first=True, dtype=int)

In [626]:
df6.sample(7)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,HeartDisease,RestingECG_Normal,RestingECG_ST,ST_Slope_Flat,ST_Slope_Up
898,35,0,ATA,122,192,0,174,0,0.0,0,1,0,0,1
689,67,1,ASY,106,223,0,142,0,0.3,0,1,0,0,1
108,50,0,ASY,140,129,0,135,0,0.0,0,1,0,0,1
37,41,1,ATA,110,250,0,142,0,0.0,0,0,1,0,1
637,43,0,ASY,115,303,0,181,0,1.2,0,1,0,1,0
603,74,0,ASY,155,310,0,112,1,1.5,1,1,0,0,0
293,65,0,ASY,115,0,0,93,1,0.0,1,1,0,1,0


In [627]:
df6

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,HeartDisease,RestingECG_Normal,RestingECG_ST,ST_Slope_Flat,ST_Slope_Up
0,40,0,ATA,140,289,0,172,0,0.0,0,1,0,0,1
1,49,1,NAP,160,180,0,156,0,1.0,1,1,0,1,0
2,37,0,ATA,130,283,0,98,0,0.0,0,0,1,0,1
3,48,1,ASY,138,214,0,108,1,1.5,1,1,0,1,0
4,54,0,NAP,150,195,0,122,0,0.0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,0,TA,110,264,0,132,0,1.2,1,1,0,1,0
914,68,0,ASY,144,193,1,141,0,3.4,1,1,0,1,0
915,57,0,ASY,130,131,0,115,1,1.2,1,1,0,1,0
916,57,1,ATA,130,236,0,174,0,0.0,1,0,0,1,0


In [628]:
ohe = OneHotEncoder(sparse_output=False, drop='first')
dfhe = pd.DataFrame(
    ohe.fit_transform(df6[['ChestPainType']]),
    columns=ohe.get_feature_names_out(['ChestPainType']),
    index=df6.index
    )
df7 = pd.concat([df6.drop(['ChestPainType'], axis=1), dfhe], axis=1)
df7.head(10)

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,HeartDisease,RestingECG_Normal,RestingECG_ST,ST_Slope_Flat,ST_Slope_Up,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA
0,40,0,140,289,0,172,0,0.0,0,1,0,0,1,1.0,0.0,0.0
1,49,1,160,180,0,156,0,1.0,1,1,0,1,0,0.0,1.0,0.0
2,37,0,130,283,0,98,0,0.0,0,0,1,0,1,1.0,0.0,0.0
3,48,1,138,214,0,108,1,1.5,1,1,0,1,0,0.0,0.0,0.0
4,54,0,150,195,0,122,0,0.0,0,1,0,0,1,0.0,1.0,0.0
5,39,0,120,339,0,170,0,0.0,0,1,0,0,1,0.0,1.0,0.0
6,45,1,130,237,0,170,0,0.0,0,1,0,0,1,1.0,0.0,0.0
7,54,0,110,208,0,142,0,0.0,0,1,0,0,1,1.0,0.0,0.0
8,37,0,140,207,0,130,1,1.5,1,1,0,1,0,0.0,0.0,0.0
9,48,1,120,284,0,120,0,0.0,0,1,0,0,1,1.0,0.0,0.0


In [629]:
df7.select_dtypes(include='object').columns

Index([], dtype='object')

In [630]:
df7.info()

<class 'pandas.core.frame.DataFrame'>
Index: 899 entries, 0 to 917
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Age                899 non-null    int64  
 1   Sex                899 non-null    int64  
 2   RestingBP          899 non-null    int64  
 3   Cholesterol        899 non-null    int64  
 4   FastingBS          899 non-null    int64  
 5   MaxHR              899 non-null    int64  
 6   ExerciseAngina     899 non-null    int64  
 7   Oldpeak            899 non-null    float64
 8   HeartDisease       899 non-null    int64  
 9   RestingECG_Normal  899 non-null    int64  
 10  RestingECG_ST      899 non-null    int64  
 11  ST_Slope_Flat      899 non-null    int64  
 12  ST_Slope_Up        899 non-null    int64  
 13  ChestPainType_ATA  899 non-null    float64
 14  ChestPainType_NAP  899 non-null    float64
 15  ChestPainType_TA   899 non-null    float64
dtypes: float64(4), int64(12)
memory

In [631]:
df8 = df7.drop('HeartDisease', axis=1)
y = df7['HeartDisease']

In [632]:
scaler = MinMaxScaler()
X = scaler.fit_transform(df8)
X

array([[0.24489796, 0.        , 0.57142857, ..., 1.        , 0.        ,
        0.        ],
       [0.42857143, 1.        , 0.76190476, ..., 0.        , 1.        ,
        0.        ],
       [0.18367347, 0.        , 0.47619048, ..., 1.        , 0.        ,
        0.        ],
       ...,
       [0.59183673, 0.        , 0.47619048, ..., 0.        , 0.        ,
        0.        ],
       [0.59183673, 1.        , 0.47619048, ..., 1.        , 0.        ,
        0.        ],
       [0.20408163, 0.        , 0.55238095, ..., 0.        , 1.        ,
        0.        ]])

In [633]:
cross_val_score(SVC(), X, y, cv=3)

array([0.87      , 0.84666667, 0.73913043])

In [634]:
cross_val_score(RandomForestClassifier(), X, y, cv=3)

array([0.85666667, 0.86333333, 0.77591973])

In [635]:
cross_val_score(LogisticRegression(), X, y, cv=3)

array([0.87333333, 0.86      , 0.76254181])

In [636]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=30)

In [637]:
model = RandomForestClassifier().fit(X_train, y_train)
model.score(X_test, y_test)

0.8444444444444444

In [638]:
X.shape

(899, 15)

In [639]:
pca = PCA(0.95)
x_pca = pca.fit_transform(X)
x_pca

array([[ 1.10538522e+00,  3.06085513e-01, -5.43762555e-01, ...,
         1.66037564e-01, -8.30579361e-02,  1.00542122e-01],
       [-1.31100743e-01,  5.97391108e-01,  9.39269190e-01, ...,
         4.15710023e-02,  6.24749014e-02,  2.02988582e-02],
       [ 8.19267750e-01, -1.02759464e+00, -5.96224007e-01, ...,
         4.85443408e-02, -6.38014613e-02,  7.19624442e-02],
       ...,
       [-8.13771319e-01,  5.99208079e-01, -2.54438755e-01, ...,
        -9.22861584e-02,  1.82359307e-02, -7.01254634e-02],
       [-5.66268284e-02, -1.58643443e-01, -2.91507882e-01, ...,
        -1.14248771e-01, -8.10214787e-04, -1.91900564e-01],
       [ 9.26791050e-01,  2.79254848e-01,  7.35457385e-01, ...,
         1.22711864e-01, -1.88829189e-01,  4.54368954e-03]])

In [640]:
X_pca_train, X_pca_test, y_train, y_test = train_test_split(x_pca, y, test_size=0.2, random_state=30)

In [641]:
model2 = RandomForestClassifier().fit(X_pca_train, y_train)
model2.score(X_pca_test, y_test)

0.8277777777777777

In [642]:
pca.explained_variance_ratio_

array([0.30145329, 0.15916859, 0.10317671, 0.09531264, 0.08864394,
       0.06821456, 0.04665943, 0.03678852, 0.02125458, 0.01976373,
       0.017587  ])

In [643]:
pca.n_components_

np.int64(11)